In [ ]:
# INQUIRE_CSI – Contaminant Source Identification Pipeline
## Part 3 – INQUIRE_CSI_Pt3_Visualization
"""
Visualizes the best cover per sample: country × source-category heatmaps, Sankey diagram and interactive 3D networks.
Documentation: `docs/03_visualization.md`
"""


In [ ]:
# Reset environment to erase previous vars, dicts, DFs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Imports & setup
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging
import hashlib
import random
from tqdm.auto import tqdm
from tqdm.auto import trange
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import beta
import openpyxl
import time
import matplotlib.pyplot as plt
from matplotlib import colormaps
from dataclasses import dataclass, field
from collections import defaultdict

import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

import json
import webbrowser

# Logging setup (equivalent to logger + appender_tee)
log_file = f"inquire_csi_pt3_visualization_log_{datetime.now().strftime('%y%m%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler(log_file, mode="w"),
        logging.StreamHandler()
    ]
)


# Helper functions
def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

def percent(n, N):
    return f"{(100 * n / N):.1f}%" if N > 0 else "0%"

# Validation helpers
def check_file(path_obj, label):
    if not path_obj.exists():
        raise FileNotFoundError(f" - {label} missing: {path_obj}")
    logging.info(f" - Found file: {label}")

def check_columns(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f" - {df_name} missing required columns: {missing}")
    logging.info(f" - {df_name}: required columns present")

# Preferred encodings (tried in order)
ENCODINGS = ["utf-8", "utf-8-sig", "cp1252", "cp1250", "latin1"]

# Read only the header row using multiple encodings. Supports CSV, TSV, XLSX and XLS.
def _read_header_only(path: Path):
    suffix = path.suffix.lower()
    if suffix in [".csv", ".tsv"]:
        sep = "\t" if suffix == ".tsv" else ","
        last_error = None
        for enc in ENCODINGS:
            try:
                df = pd.read_csv(path, nrows=0, sep=sep, encoding=enc)
                logging.info(f" - Header read using encoding '{enc}'")
                return list(df.columns)
            except UnicodeDecodeError as e:
                last_error = e
        raise RuntimeError(f"Unable to decode header using supported encodings.\nLast error:\n{last_error}")
    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, nrows=0)
        return list(df.columns)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

# Check that all expected columns exist before loading the full dataset
def check_cols_before_load(path, expected_cols, dataset_name="dataset"):
    path = Path(path)
    try:
        cols = _read_header_only(path)
    except Exception as e:
        raise RuntimeError(f"[{dataset_name}] failed to read header:\n{e}")
    missing = [c for c in expected_cols if c not in cols]
    if missing:
        raise ValueError(
            f"[{dataset_name}] missing columns:\n"
            f"{missing}\n\n"
            f"Available columns:\n{cols}"
        )
    logging.info(f" - [{dataset_name}] column check passed")
    return True

# Read CSV safely using automatic encoding fallback
def safe_read_csv(path_obj, label, **kwargs):
    if "low_memory" not in kwargs:
        kwargs["low_memory"] = False
    check_file(path_obj, label)
    last_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path_obj, encoding=enc, **kwargs)
            logging.info(
                f" - Loaded {label} "
                f"(encoding={enc}) | "
                f"rows={len(df):,} cols={len(df.columns)}"
            )
            return df
        except UnicodeDecodeError as e:
            last_error = e
    raise RuntimeError(
        f" - Failed loading {label}. "
        f"None of the tested encodings worked.\n"
        f"Last error:\n{last_error}"
    )

# Read Excel safely      
def safe_read_excel(path_obj, label, **kwargs):
    check_file(path_obj, label)
    try:
        df = pd.read_excel(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

majorsep()
logging.info(f"Imports & logging setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Check if necessary columns exist in input data files
majorsep()
logging.info(f"Checking if necessary columns exist in input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Data folder & subfolders
DATA_DIR = Path("../data") # specify path to your working folder (default: 'data' folder of the repository)
IN_DIR = Path(DATA_DIR / "set_cover_outputs")
FIG_DIR = Path(DATA_DIR / "set_cover_outputs" / "set_cover_figures")
FIG_DIR.mkdir(exist_ok=True)

# Check columns
check_cols_before_load(IN_DIR / "cover_results_all_samples_beam.csv", ["sample", "source_name", "source_general_category",
    "source_harmonized", "source_class", "source_class_domain", "P", "marginal_P", "inchikey", "feature_name",
    "n_unique_chemicals", "n_unique_sources", "n_unique_source_harmonized"], "set_cover_all_samples")

minorsep()
logging.info(f"Done checking if necessary columns exist in input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load input data
majorsep()
logging.info(f"Loading input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Read files
df_plot = safe_read_csv(IN_DIR / "cover_results_all_samples_beam.csv", "set_cover_all_samples", sep=",")
logging.info(f" - Loaded set_cover_all_samples | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Done loading input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Ensure labels are clean
majorsep()
logging.info(f"Ensuring labels are clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

df_plot["sample"] = (df_plot["sample"].astype(str).str.strip())
df_plot["source_name"] = (df_plot["source_name"].astype(str).str.strip())
df_plot["source_general_category"] = (df_plot["source_general_category"].astype(str).str.strip())
df_plot["source_harmonized"] = (df_plot["source_harmonized"].astype(str).str.strip())
df_plot["source_class"] = (df_plot["source_class"].astype(str).str.strip())
df_plot["source_class_domain"] = (df_plot["source_class_domain"].astype(str).str.strip())

logging.info(f"Ensured labels are clean | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Sample category (e.g., country) = first two characters of sample ID
logging.info(f"Creating two-letter sample codes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

df_plot["country"] = (df_plot["sample"].astype(str).str[:2])

logging.info(f"Created two-letter sample codes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Optionally drop all samples from a certain category by its 2-letter code (LL)
# DROP_LL_SAMPLES = True  # True = drop LL samples, False = keep them
DROP_LL_SAMPLES = False

if DROP_LL_SAMPLES:
    logging.info(f"Dropping selected samples | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info(f" - df_plot shape before: {df_plot.shape}")

    ll_mask = df_plot["sample"].str.upper().str.startswith("FR") # specify a 2-letter code, e.g., "FR", that matches sample IDs like 'FR01'
    ll_samples = sorted(df_plot.loc[ll_mask, "sample"].unique())
    df_plot = df_plot[~ll_mask].copy()

    logging.info(f" - Dropped {len(ll_samples)} samples: {ll_samples}")
    logging.info(f" - df_plot shape after: {df_plot.shape}")
    logging.info(f"Dropped selected samples | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    logging.info("All samples kept (DROP_LL_SAMPLES = False)")
minorsep()


# Retain only cover #1 for each sample
logging.info(f"Retaining only cover #1 for each sample | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logging.info(f" - df_plot shape before:  {df_plot.shape}")

df_plot = df_plot[df_plot["cover_id"] == 1].copy()

logging.info(f" - df_plot shape after: {df_plot.shape}")

logging.info(f"Retained only cover #1 for each sample | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build graph
majorsep()
logging.info(f"Starting building graph | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

G = nx.Graph()

for _, row in df_plot.iterrows():
    s = f"S::{row['sample']}"
    c = f"C::{row['feature_name']}"
    p = f"P::{row['source_name']}"

    # Sample node
    G.add_node(s, type="sample", country=row["country"])

    # Compound node
    G.add_node(c, type="compound")

    # Source node
    G.add_node(p,
        type="source",
        source_general_category=row["source_general_category"],
        source_harmonized=row["source_harmonized"],
        source_class=row["source_class"], # source_class
        source_class_domain=row["source_class_domain"]) # source_class_domain

    weight = 1.0

    # Explicit full connectivity
    G.add_edge(s, c, weight=weight)
    G.add_edge(c, p, weight=weight)

logging.info(f"Done starting building graph | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Precompute connection counts for hover
majorsep()
logging.info(f"Computing node connection statistics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sample_n_chem = {}
compound_n_samples = {}
compound_n_sources = {}
source_n_chem = {}

for node, data in G.nodes(data=True):
    neighbors = list(G.neighbors(node))

    if data["type"] == "sample":
        # Sample connected only to compounds
        n_chem = len(neighbors)
        sample_n_chem[node] = n_chem

    elif data["type"] == "compound":
        # Compound connected to samples and sources
        n_samples = len([n for n in neighbors if G.nodes[n]["type"] == "sample"])
        n_sources = len([n for n in neighbors if G.nodes[n]["type"] == "source"])

        compound_n_samples[node] = n_samples
        compound_n_sources[node] = n_sources

    elif data["type"] == "source":
        # Source connected only to compounds
        n_chem = len(neighbors)
        source_n_chem[node] = n_chem

logging.info(f"Node connection statistics computed | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build deterministic earth-layer spherical layout
majorsep()
logging.info(f"Building deterministic earth-layer spherical layout | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Deterministic earth-layer spherical layout
logging.info(f" - Starting building deterministic earth-layer spherical layout | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

countries = sorted(df_plot["country"].dropna().unique())
num_countries = len(countries)

if num_countries == 0:
    raise ValueError(f" - No countries found in df_plot")

# Radii (Earth analogy)
R_country  = 12   # crust
R_sample   = 10   # upper crust
R_compound = 6    # mantle
R_source   = 2.5  # core

pos = {}

# Country sector allocation (CRUST)
logging.info(f" - Allocating country sectors on outer crust | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Equal angular sectors
sector_width = 2 * np.pi / num_countries

for i, country in enumerate(countries):
    theta_center = i * sector_width
    phi_center = np.pi / 2  # equatorial ring

    x = (R_country * np.cos(theta_center) * np.sin(phi_center))
    y = (R_country * np.sin(theta_center) * np.sin(phi_center))
    z = (R_country * np.cos(phi_center))

    pos[f"COUNTRY::{country}"] = np.array([x, y, z])

# Sample dispertion within country sectors
logging.info(f" - Spreading samples within country sectors | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for i, country in enumerate(countries):
    theta_center = i * sector_width
    sector_half = (sector_width * 0.35) # leave river gaps

    samples_country = sorted(
        df_plot[
            df_plot["country"] == country
        ]["sample"]
        .unique()
    )

    n = len(samples_country)

    if n == 0:
        continue

    for j, sample in enumerate(samples_country):
        # distribute samples inside sector
        theta = (theta_center - sector_half + (2 * sector_half) * (j / max(1, n - 1)))
        phi = (np.pi / 2 + np.random.uniform(-0.15, 0.15))

        x = (R_sample * np.cos(theta) * np.sin(phi))
        y = (R_sample * np.sin(theta) * np.sin(phi))
        z = (R_sample * np.cos(phi))

        pos[f"S::{sample}"] = np.array([x, y, z])

# Compound layer (MANTLE)
logging.info(f" - Placing compounds in mantle | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

compounds = sorted(df_plot["feature_name"].dropna().unique())
n_comp = len(compounds)

if n_comp == 0:
    raise ValueError(f" - No compounds found in df_plot")

for i, compound in enumerate(compounds):
    theta = (2 * np.pi * i / n_comp)
    phi = (np.pi / 2 + 0.3 * np.sin(i))

    x = (R_compound * np.cos(theta) * np.sin(phi))
    y = (R_compound * np.sin(theta) * np.sin(phi))
    z = (R_compound * np.cos(phi))

    pos[f"C::{compound}"] = np.array([x, y, z])

# Source layer (CORE)
logging.info(f" - Placing sources in core | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

sources = sorted(df_plot["source_name"].dropna().unique())
n_source = len(sources)

if n_source == 0:
    raise ValueError(f"No sources found in df_plot")

for i, source in enumerate(sources):
    theta = (2 * np.pi * i / n_source)
    phi = (np.pi / 2 + 0.4 * np.cos(i))

    x = (R_source * np.cos(theta) * np.sin(phi))
    y = (R_source * np.sin(theta) * np.sin(phi))
    z = (R_source * np.cos(phi))

    pos[f"P::{source}"] = np.array([x, y, z])

minorsep()
logging.info(f"Done building deterministic earth-layer spherical layout | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Color maps
majorsep()
logging.info(f"Creating color maps | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Country colors
custom_colors = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#e9ed6d",  # yellow
    "#59b3ae",  # turquoise
    "#e377c2",  # pink
    "#8B0000",  # dark red
]

# Prevent IndexError if more countries than predefined colors
if len(countries) > len(custom_colors):
    custom_colors_extended = (custom_colors * (len(countries) // len(custom_colors) + 1))
else:
    custom_colors_extended = (custom_colors)

country_colors = {c: custom_colors_extended[i] for i, c in enumerate(countries)}

# Source general category or other source category colors
# source_categories = sorted(df_plot["source_general_category"].dropna().unique())
# source_categories = sorted(df_plot["source_harmonized"].dropna().unique()) # source_harmonized
# source_categories = sorted(df_plot["source_class"].dropna().unique()) # source_class
source_categories = sorted(df_plot["source_class_domain"].dropna().unique()) # source_class_domain


# Use custom colors for source categories
custom_colors_source = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # grey
    "#bcbd22",  # olive
    "#17becf",  # cyan
    "#aec7e8",  # light blue
    "#ffbb78",  # light orange
    "#98df8a",  # light green
    "#ff9896",  # salmon
    "#c5b0d5",  # lavender
    "#c49c94",  # tan
    "#f7b6d2",  # light pink
    "#c7c7c7",  # light grey
    "#dbdb8d",  # light olive
    "#9edae5",  # light cyan
]

# Prevent IndexError if more categories than predefined colors
if len(source_categories) > len(custom_colors_source):
    custom_colors_source_extended = (custom_colors_source * (len(source_categories) // len(custom_colors_source) + 1))
else:
    custom_colors_source_extended = (custom_colors_source)

source_colors = {c: custom_colors_source_extended[i] for i, c in enumerate(source_categories)}

source_category_colors = {c: custom_colors_source_extended[i] for i, c in enumerate(source_categories)}

logging.info(f"Color maps created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build plotly 3D figure
majorsep()
logging.info(f"Extracting node coordinates from layout | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

sample_x, sample_y, sample_z = [], [], []
compound_x, compound_y, compound_z = [], [], []
source_x, source_y, source_z = [], [], []

sample_text = []
compound_text = []
source_text = []

sample_colors = []
source_colors = []

# Extract node coordinates and hover information
for node, data in G.nodes(data=True):
    x, y, z = pos[node]  # always use pos

    if data["type"] == "sample":
        sample_x.append(x)
        sample_y.append(y)
        sample_z.append(z)

        sample_text.append(
            f"<b>Sample:</b> {node[3:]}<br>"
            f"<b>Country:</b> {data['country']}<br>"
            f"<b>N Chemicals:</b> {sample_n_chem[node]}"
        )

        sample_colors.append(country_colors[data["country"]])

    elif data["type"] == "compound":
        compound_x.append(x)
        compound_y.append(y)
        compound_z.append(z)

        compound_text.append(
            f"<b>Compound:</b> {node[3:]}<br>"
            f"<b>N Samples:</b> {compound_n_samples[node]}<br>"
            f"<b>N Sources:</b> {compound_n_sources[node]}"
        )

    elif data["type"] == "source":
        source_x.append(x)
        source_y.append(y)
        source_z.append(z)

        source_text.append(
            f"<b>Source:</b> {node[3:]}<br>"
            f"<b>General Category:</b> "
            f"{data.get('source_general_category')}<br>"
            f"<b>Harmonized Category:</b> "
            f"{data.get('source_harmonized')}<br>"
            f"<b>Source Class:</b> "
            f"{data.get('source_class')}<br>"
            f"<b>Source Class Domain:</b> "
            f"{data.get('source_class_domain')}<br>"            
            f"<b>N Chemicals:</b> "
            f"{source_n_chem[node]}"
        )

        source_colors.append(
            # source_colors.get(
            source_category_colors.get(                
                # unmute source category to be used for plotting
                # data.get("source_general_category"),
                # data.get("source_harmonized"), # source_harmonized
                # data.get("source_class"), # source_class
                data.get("source_class_domain"), # source_class_domain
                "#999999"
            )
        )

logging.info(f"Node coordinate extraction complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Build edges
majorsep()
logging.info(f"Starting building 3D graph with dual legends | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()
logging.info(f" - Adding edge & node traces | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Edges
edge_x, edge_y, edge_z = [], [], []

for edge in G.edges():
    x0, y0, z0 = pos[edge[0]]
    x1, y1, z1 = pos[edge[1]]

    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_z += [z0, z1, None]

edge_trace = go.Scatter3d(
    x=edge_x,
    y=edge_y,
    z=edge_z,
    mode='lines',
    line=dict(
        width=0.1,
        color='#d6d4d4'
    ),
    hoverinfo='none',
    showlegend=False
)

# Node traces
sample_trace = go.Scatter3d(
    x=sample_x,
    y=sample_y,
    z=sample_z,
    mode='markers',
    marker=dict(
        size=4,
        color=sample_colors,
        symbol='diamond'
    ),
    text=sample_text,
    hoverinfo="text",
    name="Samples",
    showlegend=False
)

compound_trace = go.Scatter3d(
    x=compound_x,
    y=compound_y,
    z=compound_z,
    mode='markers',
    marker=dict(
        size=5,
        color="#737373",
        symbol='circle'
    ),
    text=compound_text,
    hoverinfo="text",
    name="Compounds",
    showlegend=False
)

source_trace = go.Scatter3d(
    x=source_x,
    y=source_y,
    z=source_z,
    mode='markers',
    marker=dict(
        size=4,
        color=source_colors,
        symbol='square'
    ),
    text=source_text,
    hoverinfo="text",
    name="Sources",
    showlegend=False
)

fig = go.Figure(
    data=[
        edge_trace,
        sample_trace,
        compound_trace,
        source_trace
    ]
)

logging.info(f" - Added edge & node traces | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Add legend 1 — country colors (left)
logging.info(f" - Adding country legend (left side) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for country in countries:
    fig.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode='markers',
            marker=dict(
                size=6,
                color=country_colors[country],
                symbol='diamond'
            ),
            name=f"{country}",
            legend="legend",
            showlegend=True
        )
    )

# Added spacer
fig.add_trace(
    go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(
            size=6,
            color="rgba(0,0,0,0)"
        ),
        name=" ",
        legend="legend",
        showlegend=True,
    )
)

# Added chemical compounds entry
fig.add_trace(
    go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(
            size=6,
            color="#737373",
            symbol='circle'
        ),
        name="Chemical<br>Compounds",
        legend="legend",
        showlegend=True,
    )
)

logging.info(f" - Added country legend (left side) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Add legend 2 — source category colors (right)
logging.info(f" - Adding source category legend (right side) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

for src in source_categories:
    fig.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode='markers',
            marker=dict(
                size=6,
                color=source_category_colors[src],
                symbol='square'
            ),
            name=f"{src}",
            legend="legend2",
            showlegend=True
        )
    )

logging.info(f" - Added source category legend (right side) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Layout with two separate legends
logging.info(f" - Adding layout with two separate legends | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

fig.update_layout(
    # title="Contaminant Source Identification",
    margin=dict(l=0, r=0, t=50, b=0),
    scene=dict(
        aspectmode='cube',
        domain=dict(x=[0.0, 0.85], y=[0.0, 1.0]),
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
    ),

    # Left legend (countries)
    legend=dict(
        x=0.01,
        y=0.98,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=0.5,
        title="Country",
        font=dict(size=12)
    ),

    # Right legend (source category)
    legend2=dict(
        x=0.99,
        y=0.98,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=0.5,
        title="Source Category",
        # font=dict(size=12)
        font=dict(size=9)
    )
)

logging.info(f" - Added layout with two separate legends | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()
logging.info(f"Done building 3D graph with dual legends | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Display and save figure
majorsep()
logging.info(f"Saving interactive 3D Earth-style graph with dual legends | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

fig.show()

# Save HTML
fig_html = (FIG_DIR / f"inquire_csi_figure.html")
pio.write_html(fig, fig_html, include_plotlyjs="cdn")

minorsep()
logging.info(f"Interactive 3D Earth-style graph with dual legends saved | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# 3D spherical country-specific CSI networks (one interactive HTML figure per country)
# Prepare data
majorsep()
logging.info(f"Preparing country-specific CSI plots | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Work from the already prepared plotting dataframe
df_plot_all = df_plot.copy()

# Country = first two characters of sample ID
df_plot_all["country"] = df_plot_all["sample"].astype(str).str[:2].str.upper()

# Keep only cover 1
if "cover_id" in df_plot_all.columns:
    df_plot_all = df_plot_all[pd.to_numeric(df_plot_all["cover_id"], errors="coerce") == 1].copy()

logging.info(f" - {len(df_plot_all)} rows | {df_plot_all['country'].nunique()} countries")

# Country list
countries_plot = sorted(df_plot_all["country"].dropna().unique())

logging.info(f" - Countries to plot: {', '.join(countries_plot)}")
minorsep()

logging.info(f"Data prepared for country-specific plots | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Color maps
majorsep()
logging.info(f"Creating color maps | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Country colors
custom_colors = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#e9ed6d",  # yellow
    "#59b3ae",  # turquoise
    "#e377c2",  # pink
    "#8B0000",  # dark red
]

# Extend color list if needed
custom_colors_extended = custom_colors * (len(countries_plot) // len(custom_colors) + 1)
country_colors_plot = {c: custom_colors_extended[i] for i, c in enumerate(countries_plot)}

# Source general category or other source category colors
# source_categories_all = sorted(df_plot_all["source_general_category"].dropna().astype(str).unique())
# source_categories_all = sorted(df_plot_all["source_harmonized"].dropna().astype(str).unique()) # source_harmonized
# source_categories_all = sorted(df_plot_all["source_class"].dropna().astype(str).unique()) # source_class
source_categories_all = sorted(df_plot_all["source_class_domain"].dropna().astype(str).unique()) # source_class_domain

custom_colors_source = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # grey
    "#bcbd22",  # olive
    "#17becf",  # cyan
    "#aec7e8",  # light blue
    "#ffbb78",  # light orange
    "#98df8a",  # light green
    "#ff9896",  # salmon
    "#c5b0d5",  # lavender
    "#c49c94",  # tan
    "#f7b6d2",  # light pink
    "#c7c7c7",  # light grey
    "#dbdb8d",  # light olive
    "#9edae5",  # light cyan
]

# Extend color list if needed
custom_colors_source_extended = custom_colors_source * (len(source_categories_all) // len(custom_colors_source) + 1)
source_category_colors = {c: custom_colors_source_extended[i] for i, c in enumerate(source_categories_all)}

logging.info(f"Color maps created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Loop over countries
majorsep()

for country in countries_plot:
    logging.info(f"Starting country-specific plot: {country} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    majorsep()

    # Filter country
    df_country = df_plot_all[df_plot_all["country"] == country].copy()

    if df_country.empty:
        logging.warning(f"No data found for country {country} → skipping")
        continue

    logging.info(
        f"{country}: {len(df_country)} rows | "
        f"{df_country['sample'].nunique()} samples | "
        f"{df_country['feature_name'].nunique()} compounds | "
        f"{df_country['source_name'].nunique()} sources"
    )

    # Build graph
    logging.info(f"Building graph for {country} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    G_country = nx.Graph()

    for _, row in df_country.iterrows():
        s = f"S::{row['sample']}"
        c = f"C::{row['feature_name']}"
        src = f"SRC::{row['source_name']}"

        # Sample
        G_country.add_node(s, type="sample", country=row["country"])

        # Compound
        G_country.add_node(c, type="compound")

        # Source
        G_country.add_node(src,
            type="source",
            source_general_category=row.get("source_general_category", None),
            source_harmonized=row.get("source_harmonized", None),
            source_class=row.get("source_class", None),
            source_class_domain=row.get("source_class_domain", None))

        # Keep the same explicit full connectivity
        weight = 1.0

        G_country.add_edge(s, c, weight=weight)
        G_country.add_edge(c, src, weight=weight)

    logging.info(f"Graph built for {country}: {G_country.number_of_nodes()} nodes, {G_country.number_of_edges()} edges")
    minorsep()

    # Precompute connections for hover
    logging.info(f"Computing node connection statistics for {country}")

    sample_n_chem = {}
    compound_n_samples = {}
    compound_n_sources = {}
    source_n_chem = {}

    for node, data in G_country.nodes(data=True):
        neighbors = list(G_country.neighbors(node))

        if data["type"] == "sample":
            n_chem = len(neighbors)
            sample_n_chem[node] = n_chem
        elif data["type"] == "compound":
            n_samples = len([n for n in neighbors if G_country.nodes[n]["type"] == "sample"])
            n_sources = len([n for n in neighbors if G_country.nodes[n]["type"] == "source"])
            compound_n_samples[node] = n_samples
            compound_n_sources[node] = n_sources
        elif data["type"] == "source":
            n_chem = len(neighbors)
            source_n_chem[node] = n_chem

    logging.info(f"Node connection statistics computed for {country}")

    # Deterministic spherical earth-layer layout
    logging.info(f"Building spherical layout for {country}")

    # Radii
    R_country = 12
    R_sample = 10
    R_compound = 6
    R_source = 2.5

    pos_country = {}

    # Country position
    # Single country → place it at the top/equator reference position
    theta_center = 0
    phi_center = np.pi / 2

    x = (R_country * np.cos(theta_center) * np.sin(phi_center))
    y = (R_country * np.sin(theta_center) * np.sin(phi_center))
    z = R_country * np.cos(phi_center)

    pos_country[f"Country::{country}"] = np.array([x, y, z])

    # Samples — Full circular distribution
    samples_country = sorted(df_country["sample"].dropna().astype(str).unique())
    n_samples = len(samples_country)

    if n_samples == 0:
        raise ValueError(f"No samples found for country {country}")


    for j, sample in enumerate(samples_country):       
        # Evenly distribute samples over the full 360° circumference
        theta = 2 * np.pi * j / n_samples
        # Small deterministic vertical variation
        phi = np.pi / 2 + 0.15 * np.sin(2 * np.pi * j / n_samples)

        x = (R_sample * np.cos(theta) * np.sin(phi))
        y = (R_sample * np.sin(theta) * np.sin(phi))
        z = R_sample * np.cos(phi)

        pos_country[f"S::{sample}"] = np.array([x, y, z])

    # Compounds (mantle)
    compounds_country = sorted(df_country["feature_name"].dropna().astype(str).unique())
    n_comp = len(compounds_country)

    if n_comp > 0:
        for i, compound in enumerate(compounds_country):
            theta = (2 * np.pi * i / n_comp)
            phi = (np.pi / 2 + 0.3 * np.sin(i))

            x = (R_compound * np.cos(theta) * np.sin(phi))
            y = (R_compound * np.sin(theta) * np.sin(phi))
            z = R_compound * np.cos(phi)

            pos_country[f"C::{compound}"] = np.array([x, y, z])

    # Sources (core)
    sources_country = sorted(df_country["source_name"].dropna().astype(str).unique())
    n_source = len(sources_country)

    if n_source > 0:
        for i, source in enumerate(sources_country):
            theta = (2 * np.pi * i / n_source)
            phi = (np.pi / 2 + 0.4 * np.cos(i))

            x = (R_source * np.cos(theta) * np.sin(phi))
            y = (R_source * np.sin(theta) * np.sin(phi))
            z = R_source * np.cos(phi)

            pos_country[f"SRC::{source}"] = np.array([x, y, z])

    logging.info(f"Spherical layout completed for {country}")

    # Extract node coordinates
    logging.info(f"Extracting node coordinates for {country}")

    sample_x, sample_y, sample_z = [], [], []
    compound_x, compound_y, compound_z = [], [], []
    source_x, source_y, source_z = [], [], []

    sample_text = []
    compound_text = []
    source_text = []

    sample_colors = []
    source_colors_plot = []

    # Node coordinates + hover information
    for node, data in G_country.nodes(data=True):
        # Country reference node is not plotted
        if node.startswith("COUNTRY::"):
            continue

        if node not in pos_country:
            continue

        x, y, z = pos_country[node]

        if data["type"] == "sample":
            sample_x.append(x)
            sample_y.append(y)
            sample_z.append(z)

            sample_text.append(
                f"<b>Sample:</b> {node[3:]}<br>"
                f"<b>Country:</b> {data['country']}<br>"
                f"<b>N Chemicals:</b> "
                f"{sample_n_chem[node]}"
            )

            sample_colors.append(country_colors[data["country"]])

        elif data["type"] == "compound":
            compound_x.append(x)
            compound_y.append(y)
            compound_z.append(z)

            compound_text.append(
                f"<b>Compound:</b> {node[3:]}<br>"
                f"<b>N Samples:</b> "
                f"{compound_n_samples[node]}<br>"
                f"<b>N Sources:</b> "
                f"{compound_n_sources[node]}"
            )

        elif data["type"] == "source":
            source_x.append(x)
            source_y.append(y)
            source_z.append(z)

            source_text.append(
                f"<b>Source:</b> {node[5:]}<br>"
                f"<b>General Category:</b> {data.get('source_general_category')}<br>"
                f"<b>Harmonized Category:</b> {data.get('source_harmonized')}<br>"
                f"<b>Source Class:</b> {data.get('source_class')}<br>"
                f"<b>Source Class Domain:</b> {data.get('source_class_domain')}<br>"
                f"<b>N Chemicals:</b> {source_n_chem[node]}"
            )

            # Color source by GENERAL category
            source_colors_plot.append(
                source_category_colors.get(
                    # unmute source category to be used for plotting
                    # data.get("source_general_category"), # source_general_category
                    # data.get("source_harmonized"), # source_harmonized
                    # data.get("source_class"), # source_class
                    data.get("source_class_domain"), # source_class_domain
                    "#999999")
            )

    logging.info(f"Node coordinate extraction complete for {country}")

    # Edges
    edge_x, edge_y, edge_z = [], [], []

    for edge in G_country.edges():
        if (edge[0] not in pos_country or edge[1] not in pos_country):
            continue

        x0, y0, z0 = pos_country[edge[0]]
        x1, y1, z1 = pos_country[edge[1]]

        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_z += [z0, z1, None]

    edge_trace = go.Scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode="lines",
        line=dict(
            width=0.1,
            color="#d6d4d4"
        ),
        hoverinfo="none",
        showlegend=False
    )

    # Node traces
    # Samples
    sample_trace = go.Scatter3d(
        x=sample_x,
        y=sample_y,
        z=sample_z,
        mode="markers",
        marker=dict(
            size=4,
            color=sample_colors,
            symbol="diamond"
        ),
        text=sample_text,
        hoverinfo="text",
        name="Samples",
        showlegend=False
    )

    # Compounds
    compound_trace = go.Scatter3d(
        x=compound_x,
        y=compound_y,
        z=compound_z,
        mode="markers",
        marker=dict(
            size=5,
            color="#737373",
            symbol="circle"
        ),
        text=compound_text,
        hoverinfo="text",
        name="Compounds",
        showlegend=False
    )

    # Sources
    source_trace = go.Scatter3d(
        x=source_x,
        y=source_y,
        z=source_z,
        mode="markers",
        marker=dict(
            size=4,
            color=source_colors_plot,
            symbol="square"
        ),
        text=source_text,
        hoverinfo="text",
        name="Sources",
        showlegend=False
    )

    # Figure
    fig_country = go.Figure(
        data=[edge_trace,
            sample_trace,
            compound_trace,
            source_trace]
    )

    # Legend 1 — Country / node types
    logging.info(f"Adding country legend for {country}")

    # Country/sample color
    fig_country.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode="markers",
            marker=dict(
                size=6,
                color=country_colors[country],
                symbol="diamond"
            ),
            name=country,
            legend="legend",
            showlegend=True
        )
    )

    # Spacer
    fig_country.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode="markers",
            marker=dict(
                size=6,
                color="rgba(0,0,0,0)"
            ),
            name=" ",
            legend="legend",
            showlegend=True
        )
    )

    # Chemical compounds
    fig_country.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode="markers",
            marker=dict(
                size=6,
                color="#737373",
                symbol="circle"
            ),
            name="Chemical<br>Compounds",
            legend="legend",
            showlegend=True
        )
    )

    # Legend 2 — Source general categories
    logging.info(f"Adding source category legend for {country}")
    minorsep()

    # Only categories actually present in this country
    # Unmute source category to be used for plotting
    # country_source_categories = sorted(df_country["source_general_category"].dropna().astype(str).unique()) # source_general_category
    # country_source_categories = sorted(df_country["source_harmonized"].dropna().astype(str).unique()) # source_harmonized
    # country_source_categories = sorted(df_country["source_class"].dropna().astype(str).unique()) # source_class
    country_source_categories = sorted(df_country["source_class_domain"].dropna().astype(str).unique()) # source_class_domain

    for src_category in country_source_categories:
        fig_country.add_trace(
            go.Scatter3d(
                x=[None],
                y=[None],
                z=[None],
                mode="markers",
                marker=dict(
                    size=6,
                    color=source_category_colors.get(
                        src_category,
                        "#999999"
                    ),
                    symbol="square"
                ),
                name=src_category,
                legend="legend2",
                showlegend=True
            )
        )

    # Layout
    fig_country.update_layout(
        # title=(f"Contaminant Source Identification — {country}"),
        margin=dict(l=0, r=0, t=50, b=0),
        scene=dict(
            aspectmode="cube",
            domain=dict(x=[0.0, 0.85], y=[0.0, 1.0]),
            xaxis=dict(showbackground=False, visible=False),
            yaxis=dict(showbackground=False, visible=False),
            zaxis=dict(showbackground=False, visible=False)
        ),
        # Left legend
        legend=dict(
            x=0.01,
            y=0.98,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="black",
            borderwidth=0.5,
            title=f"Country: {country}",
            font=dict(size=12)
        ),
        # Right legend
        legend2=dict(
            x=0.99,
            y=0.98,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="black",
            borderwidth=0.5,
            title="Source Category",
            # font=dict(size=12)
            font=dict(size=9)
        )
    )
    # # Display
    # fig_country.show()
    
    # Save HTML
    fig_country_html = (FIG_DIR / f"inquire_csi_figure_{country}.html")
    pio.write_html(fig_country, fig_country_html, include_plotlyjs="cdn")
    logging.info(f"Interactive 3D graph saved: {fig_country_html}")
    majorsep()

logging.info(f"All country-specific CSI plots completed | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Statistics of df_plot
majorsep()
logging.info(f"Calculating df_plot statistics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

df_stats = df_plot.copy()

# Country = first two characters of sample ID
df_stats["country"] = df_stats["sample"].astype(str).str[:2].str.upper()

# Keep only cover 1
if "cover_id" in df_stats.columns:
    df_stats = df_stats[pd.to_numeric(df_stats["cover_id"], errors="coerce") == 1].copy()

# Overall statistics
n_samples = df_stats["sample"].nunique()
n_compounds = df_stats["feature_name"].nunique()
n_sources = df_stats["source_name"].nunique()
n_source_categories = df_stats["source_general_category"].nunique()
n_source_harmonized = df_stats["source_harmonized"].nunique()
n_source_class = df_stats["source_class"].nunique()
n_source_class_domain = df_stats["source_class_domain"].nunique()
n_countries = df_stats["country"].nunique()

logging.info(f" - Unique countries: {n_countries}")
logging.info(f" - Unique samples: {n_samples}")
logging.info(f" - Unique compounds: {n_compounds}")
logging.info(f" - Unique sources: {n_sources}")
logging.info(f" - Unique source general categories: {n_source_categories}")
logging.info(f" - Unique source harmonized categories: {n_source_harmonized}")
logging.info(f" - Unique source classes: {n_source_class}")
logging.info(f" - Unique source class domains: {n_source_class_domain}")
minorsep()

# Country-level statistics
country_stats = (
    df_stats.groupby("country")
    .agg(
        n_samples=("sample", "nunique"),
        n_compounds=("feature_name", "nunique"),
        n_sources=("source_name", "nunique"),
        n_source_categories=("source_general_category", "nunique"),
        n_source_harmonized=("source_harmonized", "nunique"),
        n_source_class=("source_class", "nunique"),
        n_source_class_domain=("source_class_domain", "nunique"),
    )
    .sort_index()
)

logging.info(f"Country-level statistics:")
display(country_stats)

minorsep()
logging.info(f"Statistics calculated | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Country × Source General Category heatmap (portrait orientation)
majorsep()
logging.info(f"Building country × source-category heatmap (portrait orientation) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

df_heat = df_plot.copy()

# Country = first two characters of sample ID
df_heat["country"] = df_heat["sample"].astype(str).str[:2].str.upper()

# Keep only cover 1
if "cover_id" in df_heat.columns:
    df_heat = df_heat[pd.to_numeric(df_heat["cover_id"], errors="coerce") == 1].copy()

# Remove missing values required for the analysis
# Unmute source category to be used for plotting
# df_heat = df_heat.dropna(subset=["country", "feature_name", "source_general_category"]).copy() # source_general_category
# df_heat = df_heat.dropna(subset=["country", "feature_name", "source_harmonized"]).copy() # source_harmonized
# df_heat = df_heat.dropna(subset=["country", "feature_name", "source_class"]).copy() # source_class
df_heat = df_heat.dropna(subset=["country", "feature_name", "source_class_domain"]).copy() # source_class_domain

logging.info(
    f"Heatmap dataset: {len(df_heat)} rows | "
    f"{df_heat['country'].nunique()} countries | "
    f"{df_heat['feature_name'].nunique()} compounds | "
    # unmute source category to be used for plotting
    # f"{df_heat['source_general_category'].nunique()} source categories" # source_general_category
    # f"{df_heat['source_harmonized'].nunique()} source categories" # source_harmonized
    # f"{df_heat['source_class'].nunique()} source classes" # source_class
    f"{df_heat['source_class_domain'].nunique()} source class domains" # source_class_domain
)
minorsep()

# Count unique compounds per country × source category
compound_counts = (
    # unmute source category to be used for plotting
    # df_heat.groupby(["country", "source_general_category"])["feature_name"] # source_general_category
    # df_heat.groupby(["country", "source_harmonized"])["feature_name"] # source_harmonized
    # df_heat.groupby(["country", "source_class"])["feature_name"] # source_class
    df_heat.groupby(["country", "source_class_domain"])["feature_name"] # source_class_domain
    .nunique()
    .unstack(fill_value=0)
)

# Total unique compounds per country
country_compound_totals = df_heat.groupby("country")["feature_name"].nunique()

# Percentage of unique compounds attributed to each category
compound_percent = compound_counts.div(country_compound_totals, axis=0).mul(100)

# Remove very rare source categories
# Keep categories represented by at least 1% of compounds in at least one country
categories_to_keep = compound_percent.columns[(compound_percent.max(axis=0) >= 1)]
compound_counts = compound_counts[categories_to_keep]
compound_percent = compound_percent[categories_to_keep]

# Order countries
country_order = sorted(compound_percent.index)
compound_counts = compound_counts.reindex(country_order)
compound_percent = compound_percent.reindex(country_order)

# Order source categories by overall importance
category_order = compound_counts.sum(axis=0).sort_values(ascending=False).index
compound_counts = compound_counts[category_order]
compound_percent = compound_percent[category_order]

logging.info(f"Heatmap contains {len(country_order)} countries × {len(category_order)} source categories")

# Build annotation labels
annotations = compound_counts.copy().astype(str)

for country in annotations.index:
    for category in annotations.columns:
        n = compound_counts.loc[country, category]
        pct = compound_percent.loc[country, category]
        annotations.loc[country, category] = f"{n}\n({pct:.1f}%)"

# Transpose
heatmap_data = compound_percent.T
heatmap_annotations = annotations.T

logging.info(f"Rotated heatmap matrix: {heatmap_data.shape[0]} source categories × {heatmap_data.shape[1]} countries")
minorsep()

# Plot
# Change figure size based on the number of source categories
fig, ax = plt.subplots(figsize=(7, 12))

im = ax.imshow(heatmap_data.values, aspect="auto", interpolation="nearest")

# Countries on top
ax.set_xticks(np.arange(len(country_order)))
ax.set_xticklabels(country_order, fontsize=10)
ax.xaxis.tick_top()

# Source categories on right
ax.set_yticks(np.arange(len(category_order)))
ax.set_yticklabels(category_order, fontsize=9)
ax.yaxis.tick_right()

# Cell annotations with adaptive text color
cmap = im.get_cmap()
norm = im.norm

for i in range(len(category_order)):
    for j in range(len(country_order)):
        value = heatmap_data.iloc[i, j]
        text = heatmap_annotations.iloc[i, j]

        # Get actual RGBA color of the heatmap cell
        rgba = cmap(norm(value))

        # Perceived luminance
        luminance = (
            0.2126 * rgba[0] +
            0.7152 * rgba[1] +
            0.0722 * rgba[2]
        )

        # Dark text on light cells, white text on dark cells
        text_color = "black" if luminance > 0.55 else "white"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=6,
            color=text_color,
            linespacing=0.9
        )

# Colorbar on the left — narrower than default
cbar = fig.colorbar(
    im,
    ax=ax,
    location="left",
    pad=0.04,
    fraction=0.023
)

# cbar.set_label("% of unique compounds attributed", fontsize=10)
cbar.set_label("% of unique compounds explained by sources\nof this category (non-exclusive)", fontsize=10)

# Clean appearance
ax.tick_params(axis="both", which="both", length=0)

fig.tight_layout()

# Save
fig_heatmap = FIG_DIR / "inquire_csi_country-source_category_heatmap.png"
fig.savefig(fig_heatmap, dpi=600, bbox_inches="tight")

# Display
plt.show()

minorsep()
logging.info(f"Heatmap (portrait orientation) saved: \n{fig_heatmap} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Country × Source General Category heatmap (landscape orientation)
majorsep()
logging.info(f"Building country × source-category heatmap (landscape orientation) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

df_heat_land = df_plot.copy()

# Country = first two characters of sample ID
df_heat_land["country"] = df_heat_land["sample"].astype(str).str[:2].str.upper()

# Keep only cover 1
if "cover_id" in df_heat_land.columns:
    df_heat_land = df_heat_land[pd.to_numeric(df_heat_land["cover_id"], errors="coerce") == 1].copy()

# Remove missing values required for the analysis
# Unmute source category to be used for plotting
# df_heat_land = df_heat_land.dropna(subset=["country", "feature_name", "source_general_category"]).copy() # source_general_category
# df_heat_land = df_heat_land.dropna(subset=["country", "feature_name", "source_harmonized"]).copy() # source_harmonized
# df_heat_land = df_heat_land.dropna(subset=["country", "feature_name", "source_class"]).copy() # source_class
df_heat_land = df_heat_land.dropna(subset=["country", "feature_name", "source_class_domain"]).copy() # source_class_domain

logging.info(
    f"Heatmap dataset: {len(df_heat_land)} rows | "
    f"{df_heat_land['country'].nunique()} countries | "
    f"{df_heat_land['feature_name'].nunique()} compounds | "
    # unmute source category to be used for plotting
    # f"{df_heat_land['source_general_category'].nunique()} source categories" # source_general_category
    # f"{df_heat_land['source_harmonized'].nunique()} source categories" # source_harmonized
    # f"{df_heat_land['source_class'].nunique()} source categories" # source_class
    f"{df_heat_land['source_class_domain'].nunique()} source categories" # source_class_domain
)
minorsep()

# Count unique compounds per country × source category
compound_counts = (
    # unmute source category to be used for plotting
    # df_heat_land.groupby(["country", "source_general_category"])["feature_name"] # source_general_category
    # df_heat_land.groupby(["country", "source_harmonized"])["feature_name"] # source_harmonized
    # df_heat_land.groupby(["country", "source_class"])["feature_name"] # source_class
    df_heat_land.groupby(["country", "source_class_domain"])["feature_name"] # source_class_domain
    .nunique()
    .unstack(fill_value=0)
)

# Total unique compounds per country
country_compound_totals = df_heat_land.groupby("country")["feature_name"].nunique()

# Percentage of unique compounds attributed to each category
compound_percent = compound_counts.div(country_compound_totals, axis=0).mul(100)

# Remove very rare source categories
# Keep categories represented by at least 1% of compounds in at least one country
categories_to_keep = compound_percent.columns[(compound_percent.max(axis=0) >= 1)]
compound_counts = compound_counts[categories_to_keep]
compound_percent = compound_percent[categories_to_keep]

# Order countries
country_order = sorted(compound_percent.index)
compound_counts = compound_counts.reindex(country_order)
compound_percent = compound_percent.reindex(country_order)

# Order source categories by overall importance
category_order = compound_counts.sum(axis=0).sort_values(ascending=False).index
compound_counts = compound_counts[category_order]
compound_percent = compound_percent[category_order]

logging.info(f"Heatmap contains {len(country_order)} countries × {len(category_order)} source categories")

# Build two-line annotation labels
annotations = compound_counts.copy().astype(str)

for country in annotations.index:
    for category in annotations.columns:
        n = compound_counts.loc[country, category]
        pct = compound_percent.loc[country, category]
        annotations.loc[country, category] = f"{n}\n{pct:.1f}%"

# Original landscape orientation
heatmap_data = compound_percent
heatmap_annotations = annotations

logging.info(f"Heatmap matrix: {heatmap_data.shape[0]} countries × {heatmap_data.shape[1]} source categories")
minorsep()

# Plot — wider landscape format for PowerPoint
# Change figure size based on the number of source categories
fig, ax = plt.subplots(figsize=(16, 5), constrained_layout=True)

im = ax.imshow(heatmap_data.values, aspect="auto", interpolation="nearest")

# Countries on left
ax.set_yticks(np.arange(len(country_order)))
ax.set_yticklabels(country_order, fontsize=12)

# Source categories on bottom — perpendicular to heatmap
ax.set_xticks(np.arange(len(category_order)))
ax.set_xticklabels(category_order, rotation=90, ha="center", va="top", fontsize=12)

# Adaptive annotation colors
cmap = im.get_cmap()
norm = im.norm

for i in range(len(country_order)):
    for j in range(len(category_order)):
        value = heatmap_data.iloc[i, j]
        text = heatmap_annotations.iloc[i, j]

        # Actual heatmap cell color
        rgba = cmap(norm(value))

        # Perceived luminance
        luminance = (
            0.2126 * rgba[0] +
            0.7152 * rgba[1] +
            0.0722 * rgba[2]
        )

        # Black on light cells, white on dark cells
        text_color = "black" if luminance > 0.55 else "white"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=7,
            color=text_color,
            fontweight="normal",
            linespacing=0.85
        )

# Colorbar on the right — narrow
cbar = fig.colorbar(
    im,
    ax=ax,
    location="right",
    pad=0.015,
    fraction=0.018
)

# cbar.set_label("% of unique compounds attributed", fontsize=10)
cbar.set_label("% of unique compounds explained by sources\nof this category (non-exclusive)", fontsize=10)
cbar.ax.tick_params(labelsize=8)

# Clean appearance
ax.tick_params(axis="both", which="both", length=0)

# Remove unnecessary axis labels
ax.set_xlabel("")
ax.set_ylabel("")

# Save high-resolution PNG
fig_heatmap = FIG_DIR / "inquire_csi_country-source_category_heatmap_land.png"
fig.savefig(fig_heatmap, dpi=600, bbox_inches="tight")

# Display
plt.show()

minorsep()
logging.info(f"Heatmap (landscape orientation) saved: \n{fig_heatmap} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Country × Source General Category Sankey diagram
majorsep()
logging.info(f"Building country × source-category Sankey diagram | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

df_sankey = df_plot.copy()

# Country names
# Specify 2-letter country codes & full country names as in the example
country_name_map = {
    "FR": "France",
    "ES": "Spain",
    "US": "United States",
    "SK": "Slovakia",
    "NZ": "New Zealand",
    "CA": "Canada",
    "PL": "Poland",
    "FI": "Finland",
}

# Country = first two characters of sample ID
df_sankey["country"] = (df_sankey["sample"].astype(str).str[:2].str.upper())

# Filter data
# Keep only cover 1
if "cover_id" in df_sankey.columns:
    df_sankey = df_sankey[
        pd.to_numeric(
            df_sankey["cover_id"],
            errors="coerce"
        ) == 1
    ].copy()

# Remove missing values required for the analysis
# Unmute source category to be used for plotting
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_general_category"]).copy() # source_general_category
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_harmonized"]).copy() # source_harmonized
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_class"]).copy() # source_class
df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_class_domain"]).copy() # source_class_domain

logging.info(
    f"Sankey dataset: {len(df_sankey)} rows | "
    f"{df_sankey['country'].nunique()} countries | "
    f"{df_sankey['feature_name'].nunique()} compounds | "
    # unmute source category to be used for plotting
    # f"{df_sankey['source_general_category'].nunique()} source categories" # source_general_category
    # f"{df_sankey['source_harmonized'].nunique()} source categories" # source_harmonized
    # "{df_sankey['source_class'].nunique()} source categories" # source_class
    f"{df_sankey['source_class_domain'].nunique()} source categories" # source_class_domain
)
minorsep()

# Count unique compounds per country × source category
compound_counts = (
    df_sankey
    .groupby(
        [
            "country",
            # unmute source category to be used for plotting
            # "source_general_category" # source_general_category
            # "source_harmonized" # source_harmonized
            # "source_class" # source_class
            "source_class_domain" # source_class_domain
        ]
    )["feature_name"]
    .nunique()
    .unstack(fill_value=0)
)

# Total unique compounds per country
country_compound_totals = (
    df_sankey
    .groupby("country")["feature_name"]
    .nunique()
)

# Percentage of unique compounds attributed to each category
compound_percent = (
    compound_counts
    .div(country_compound_totals, axis=0)
    .mul(100)
)

# Keep source categories represented by ≥1% in at least one country
categories_to_keep = compound_percent.columns[compound_percent.max(axis=0) >= 1]

logging.info(f"Retaining {len(categories_to_keep)} source categories with ≥1% representation in at least one country")

# Collapse rare source categories into "Other"
df_sankey["source_category_plot"] = (
    # unmute source category to be used for plotting
    # df_sankey["source_general_category"] # source_general_category
    # df_sankey["source_harmonized"] # source_harmonized
    # df_sankey["source_class"] # source_class
    df_sankey["source_class_domain"]  # source_class_domain
    .where(
        # unmute source category to be used for plotting
        # df_sankey["source_general_category"].isin( # source_general_category
        # df_sankey["source_harmonized"].isin( # source_harmonized
        # df_sankey["source_class"].isin( # source_class
        df_sankey["source_class_domain"].isin( # source_class_domain
            categories_to_keep
        ),
        "Other"
    )
)

# Recalculate unique compound counts after collapsing rare categories
sankey_counts = (
    df_sankey
    .groupby(
        [
            "country",
            "source_category_plot"
        ]
    )["feature_name"]
    .nunique()
    .reset_index(name="n_compounds")
)

# Country totals
country_totals = (df_sankey.groupby("country")["feature_name"].nunique().to_dict())

# Percentage within each country
sankey_counts["percentage"] = (
    sankey_counts["n_compounds"]
    / sankey_counts["country"].map(country_totals)
    * 100
)

# Country order
country_order = sorted(sankey_counts["country"].unique())

# Source category order
source_order = (
    sankey_counts
    .groupby("source_category_plot")["n_compounds"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

logging.info(f"Sankey matrix: {len(country_order)} countries × {len(source_order)} source categories")
minorsep()

# Build Sankey nodes
# Country nodes first
country_nodes = country_order

# Source category nodes second
source_nodes = source_order

# Hide native Plotly node labels
# Labels will be added as annotations so their positions can be controlled precisely
node_labels = ([""] * len(country_nodes) + [""] * len(source_nodes))

# Node indices
country_idx = {country: i for i, country in enumerate(country_nodes)}
source_idx = {source: len(country_nodes) + i for i, source in enumerate(source_nodes)}

# Node custom data
node_customdata = []

# Full country names for country-node hovers
for country in country_nodes:
    node_customdata.append(
        [country,
            country_name_map.get(
                country,
                country
            )]
    )

# Source category names for source-node hovers
for source in source_nodes:
    node_customdata.append(
        [source,
            source]
    )

# Build Sankey links
link_source = []
link_target = []
link_value = []
link_customdata = []
link_hovertemplate = []

for _, row in sankey_counts.iterrows():
    country = row["country"]
    source_category = row["source_category_plot"]
    n_compounds = int(row["n_compounds"])
    percentage = row["percentage"]

    link_source.append(country_idx[country])
    link_target.append(source_idx[source_category])
    link_value.append(n_compounds)

    full_country_name = country_name_map.get(country, country)

    link_customdata.append(
        [full_country_name,
            source_category,
            n_compounds,
            percentage]
    )

    link_hovertemplate.append(
        "<b>%{customdata[0]}</b> → "
        "<b>%{customdata[1]}</b><br>"
        "Unique compounds: %{customdata[2]}<br>"
        "Share of country's compounds (non-exclusive): "
        "%{customdata[3]:.1f}%"
        "<extra></extra>"
    )

logging.info(f"Created {len(link_value)} Sankey links")
minorsep()

# Node colors
# Country node colors
node_colors = [country_colors[country] for country in country_nodes]

# Source category node colors
node_colors.extend(source_category_colors.get(source, "#999999") for source in source_nodes)

# Fixed node positions
n_countries = len(country_nodes)
n_sources = len(source_nodes)

# Horizontal positions
country_x = 0.08
source_x = 0.92

# Vertical positions
# Plotly Sankey node.y coordinates and layout annotation paper coordinates have opposite vertical orientations
# Therefore:
#  - annotation_y = top-to-bottom visual position
#  - node_y       = 1 - annotation_y
# This keeps the rectangles and their labels aligned
top_margin = 0.04
bottom_margin = 0.96

if n_countries == 1:
    country_annotation_y = [0.5]
else:
    country_annotation_y = np.linspace(top_margin, bottom_margin, n_countries).tolist()

if n_sources == 1:
    source_annotation_y = [0.5]
else:
    source_annotation_y = np.linspace(top_margin, bottom_margin, n_sources).tolist()

# Invert the y-coordinates for Sankey nodes
country_node_y = [1 - y for y in country_annotation_y]
source_node_y = [1 - y for y in source_annotation_y]

# Combined node positions
node_x = ([country_x] * n_countries + [source_x] * n_sources)
node_y = (country_node_y + source_node_y)

logging.info(f"Fixed Sankey node positions: country x={country_x}, source x={source_x}")
minorsep()

# Create interactive Sankey figure
fig_sankey = go.Figure(
    data=[
        go.Sankey(
            arrangement="fixed",
            valueformat="d",

            node=dict(
                pad=18,
                thickness=18,

                line=dict(
                    color="rgba(80,80,80,0.5)",
                    width=0.5
                ),

                # Native labels hidden
                label=node_labels,

                color=node_colors,

                x=node_x,
                y=node_y,

                customdata=node_customdata,

                hovertemplate=(
                    "<b>%{customdata[1]}</b><br>"
                    # "Total unique compounds: %{value}"
                    "Compound–category links (non-exclusive sum): %{value}"
                    "<extra></extra>"
                )
            ),

            link=dict(
                source=link_source,
                target=link_target,
                value=link_value,
                color="rgba(100,100,100,0.1)",
                customdata=link_customdata,
                hovertemplate=link_hovertemplate
            )
        )
    ]
)

# Add country labels to the left of country rectangles
for country, y in zip(country_nodes, country_annotation_y):
    full_country_name = country_name_map.get(country, country)

    fig_sankey.add_annotation(
        x=country_x - 0.015,
        y=y,
        xref="paper",
        yref="paper",
        text=full_country_name,
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=12
        )
    )

# Add source category labels to the RIGHT of source rectangles
for source, y in zip(source_nodes, source_annotation_y):
    fig_sankey.add_annotation(
        x=source_x + 0.015,
        y=y - 0.0025,
        xref="paper",
        yref="paper",
        text=source,
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        align="left",
        font=dict(size=11)
        # font=dict(size=9) # source harmonized
    )

# Layout
fig_sankey.update_layout(
    width=1600,
    height=1200,
    margin=dict(l=170, r=250, t=30, b=30),
    font=dict(size=12),
    hoverlabel=dict(font_size=12)
)

# Save interactive HTML
fig_sankey_html = (FIG_DIR / "inquire_csi_country-source_sankey.html")
pio.write_html(fig_sankey, fig_sankey_html, include_plotlyjs="cdn")

logging.info(f"Interactive HTML Sankey diagram saved: \n{fig_sankey_html}")
minorsep()

# Display
fig_sankey.show()

minorsep()
logging.info(f"Sankey diagram saved: \n{fig_sankey_html} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Country × Source General Category Sankey diagram
majorsep()
logging.info(f"Building country × source-category Sankey diagram | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

df_sankey = df_plot.copy()

# Country names
# Specify 2-letter country codes & full country names as in the example
country_name_map = {
    "FR": "France",
    "ES": "Spain",
    "US": "United States",
    "SK": "Slovakia",
    "NZ": "New Zealand",
    "CA": "Canada",
    "PL": "Poland",
    "FI": "Finland",
}

# Country = first two characters of sample ID
df_sankey["country"] = (df_sankey["sample"].astype(str).str[:2].str.upper())

# Filter data
# Keep only cover 1
if "cover_id" in df_sankey.columns:
    df_sankey = df_sankey[
        pd.to_numeric(
            df_sankey["cover_id"],
            errors="coerce"
        ) == 1
    ].copy()

# Remove missing values required for the analysis
# Unmute source category to be used for plotting
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_general_category"]).copy() # source_general_category
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_harmonized"]).copy() # source_harmonized
# df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_class"]).copy() # source_class
df_sankey = df_sankey.dropna(subset=["country", "feature_name", "source_class_domain"]).copy() # source_class_domain

logging.info(
    f"Sankey dataset: {len(df_sankey)} rows | "
    f"{df_sankey['country'].nunique()} countries | "
    f"{df_sankey['feature_name'].nunique()} compounds | "
    # unmute source category to be used for plotting
    # f"{df_sankey['source_general_category'].nunique()} # source_general_category
    # f"{df_sankey['source_harmonized'].nunique()} source categories" # source_harmonized
    # f"{df_sankey['source_class'].nunique()} source classes" # source_class
    f"{df_sankey['source_class_domain'].nunique()} source class domains" # source_class_domain
)
minorsep()

# Count unique compounds per country × source category
compound_counts = (
    df_sankey
    .groupby(
        [
            "country",
            # unmute source category to be used for plotting
            # "source_general_category" # source_general_category
            # "source_harmonized" # source_harmonized
            # "source_class" # source_class
            "source_class_domain" # source_class_domain
        ]
    )["feature_name"]
    .nunique()
    .unstack(fill_value=0)
)

# Total unique compounds per country
country_compound_totals = (
    df_sankey
    .groupby("country")["feature_name"]
    .nunique()
)

# Percentage of unique compounds attributed to each category
compound_percent = (
    compound_counts
    .div(country_compound_totals, axis=0)
    .mul(100)
)

# Keep source categories represented by ≥1% in at least one country
categories_to_keep = compound_percent.columns[compound_percent.max(axis=0) >= 1]

logging.info(f"Retaining {len(categories_to_keep)} source categories with ≥1% representation in at least one country")

# Collapse rare source categories into "Other"
df_sankey["source_category_plot"] = (
    # unmute source category to be used for plotting
    # df_sankey["source_general_category"] # source_general_category
    # df_sankey["source_harmonized"] # source_harmonized
    # df_sankey["source_class"] # source_class
    df_sankey["source_class_domain"] # source_class_domain
    .where(
        # unmute source category to be used for plotting
        # df_sankey["source_general_category"].isin( # source_general_category
        # df_sankey["source_harmonized"].isin( # source_harmonized
        # df_sankey["source_class"].isin( # source_class
        df_sankey["source_class_domain"].isin( # source_class_domain
            categories_to_keep
        ),
        "Other"
    )
)

# Recalculate unique compound counts after collapsing rare categories
sankey_counts = (
    df_sankey
    .groupby(
        [
            "country",
            "source_category_plot"
        ]
    )["feature_name"]
    .nunique()
    .reset_index(name="n_compounds")
)

# Country totals
country_totals = (df_sankey.groupby("country")["feature_name"].nunique().to_dict())

# Percentage within each country
sankey_counts["percentage"] = (
    sankey_counts["n_compounds"]
    / sankey_counts["country"].map(country_totals)
    * 100
)

# Country order
country_order = sorted(sankey_counts["country"].unique())

# Source category order
source_order = (
    sankey_counts
    .groupby("source_category_plot")["n_compounds"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

logging.info(f"Sankey matrix: {len(country_order)} countries × {len(source_order)} source categories")
minorsep()

# Build Sankey nodes
# Country nodes first
country_nodes = country_order

# Source category nodes second
source_nodes = source_order

# Hide native Plotly node labels
# Labels will be added as annotations so their positions can be controlled precisely
node_labels = ([""] * len(country_nodes) + [""] * len(source_nodes))

# Node indices
country_idx = {country: i for i, country in enumerate(country_nodes)}
source_idx = {source: len(country_nodes) + i for i, source in enumerate(source_nodes)}

# Node custom data
node_customdata = []

# Full country names for country-node hovers
for country in country_nodes:
    node_customdata.append(
        [country,
            country_name_map.get(
                country,
                country
            )]
    )

# Source category names for source-node hovers
for source in source_nodes:
    node_customdata.append(
        [source,
            source]
    )

# Build Sankey links
link_source = []
link_target = []
link_value = []
link_customdata = []
link_hovertemplate = []

for _, row in sankey_counts.iterrows():
    country = row["country"]
    source_category = row["source_category_plot"]
    n_compounds = int(row["n_compounds"])
    percentage = row["percentage"]

    link_source.append(country_idx[country])
    link_target.append(source_idx[source_category])
    link_value.append(n_compounds)

    full_country_name = country_name_map.get(country, country)

    link_customdata.append(
        [full_country_name,
            source_category,
            n_compounds,
            percentage]
    )

    link_hovertemplate.append(
        "<b>%{customdata[0]}</b> → "
        "<b>%{customdata[1]}</b><br>"
        "Unique compounds: %{customdata[2]}<br>"
        # "Share of country's compounds: "
        "Share of country's compounds (non-exclusive): "
        "%{customdata[3]:.1f}%"
        "<extra></extra>"
    )

logging.info(f"Created {len(link_value)} Sankey links")
minorsep()

# Node colors
# Country node colors
node_colors = [country_colors[country] for country in country_nodes]

# Source category node colors
node_colors.extend(source_category_colors.get(source, "#999999") for source in source_nodes)

# Fixed node positions
n_countries = len(country_nodes)
n_sources = len(source_nodes)

# Horizontal positions
country_x = 0.08
source_x = 0.92

# Leader-line settings
SOURCE_LABEL_OFFSET = 0.04  # gap between source rectangles and their labels, paper units; was 0.015
LEADER_LINE_COLOR = "rgba(110,110,110,0.7)"
LEADER_LINE_WIDTH = 0.8
OPEN_HTML_IN_BROWSER = True  # lines are visible in the saved HTML, not in the JupyterLab preview

# Vertical positions
# Plotly Sankey node.y coordinates and layout annotation paper coordinates have opposite vertical orientations
# Therefore:
#  - annotation_y = top-to-bottom visual position
#  - node_y       = 1 - annotation_y
# This keeps the rectangles and their labels aligned
top_margin = 0.04
bottom_margin = 0.96

if n_countries == 1:
    country_annotation_y = [0.5]
else:
    country_annotation_y = np.linspace(top_margin, bottom_margin, n_countries).tolist()

if n_sources == 1:
    source_annotation_y = [0.5]
else:
    source_annotation_y = np.linspace(top_margin, bottom_margin, n_sources).tolist()

# Invert the y-coordinates for Sankey nodes
country_node_y = [1 - y for y in country_annotation_y]
source_node_y = [1 - y for y in source_annotation_y]

# Combined node positions
node_x = ([country_x] * n_countries + [source_x] * n_sources)
node_y = (country_node_y + source_node_y)

logging.info(f"Fixed Sankey node positions: country x={country_x}, source x={source_x}")
minorsep()

# Create interactive Sankey figure
fig_sankey = go.Figure(
    data=[
        go.Sankey(
            arrangement="fixed",
            valueformat="d",

            node=dict(
                pad=18,
                thickness=18,

                line=dict(
                    color="rgba(80,80,80,0.5)",
                    width=0.5
                ),

                # Native labels hidden
                label=node_labels,

                color=node_colors,

                x=node_x,
                y=node_y,

                customdata=node_customdata,

                hovertemplate=(
                    "<b>%{customdata[1]}</b><br>"
                    "Compound–category links (non-exclusive sum): %{value}"
                    "<extra></extra>"
                )
            ),

            link=dict(
                source=link_source,
                target=link_target,
                value=link_value,
                color="rgba(100,100,100,0.1)",
                customdata=link_customdata,
                hovertemplate=link_hovertemplate
            )
        )
    ]
)

# Add country labels to the left of country rectangles
for country, y in zip(country_nodes, country_annotation_y):
    full_country_name = country_name_map.get(country, country)

    fig_sankey.add_annotation(
        x=country_x - 0.015,
        y=y,
        xref="paper",
        yref="paper",
        text=full_country_name,
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=12
        )
    )

# Add source category labels to the RIGHT of source rectangles
source_label_y = []  # label positions, reused for the leader lines
for source, y in zip(source_nodes, source_annotation_y):
    source_label_y.append(y - 0.0025)
    fig_sankey.add_annotation(
        # x=source_x + 0.015,
        x=source_x + SOURCE_LABEL_OFFSET,  # room for leader lines
        y=y - 0.0025,
        xref="paper",
        yref="paper",
        text=source,
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        align="left",
        font=dict(size=11)
    )

# Leader lines: drawn in the browser after rendering, from each rectangle's REAL position to its label
leader_js = """
(function () {
  var gd = document.getElementById('{plot_id}');
  if (!gd) return;
  var srcColors = __SRC_COLORS__;
  var labelYs = __LABEL_YS__;
  var labelX = __LABEL_X__;
  var lineStyle = {color: '__LINE_COLOR__', width: __LINE_WIDTH__};
  function toRgb(c) {
    var d = document.createElement('div'); d.style.color = c; document.body.appendChild(d);
    var r = getComputedStyle(d).color; document.body.removeChild(d);
    return r.replace(/\\s+/g, '');
  }
  function drawLeaders() {
    if (gd.__leadersDone) return;
    var svg = gd.querySelector('.main-svg');
    var rects = Array.prototype.slice.call(gd.querySelectorAll('.sankey-node .node-rect'));
    if (!svg || !rects.length || !gd._fullLayout) return;
    var sz = gd._fullLayout._size;
    var svgBox = svg.getBoundingClientRect();
    var midX = svgBox.left + sz.l + sz.w / 2;
    var right = rects.filter(function (r) {
      var b = r.getBoundingClientRect(); return (b.left + b.right) / 2 > midX;
    });
    if (right.length < labelYs.length) return;
    var used = right.map(function () { return false; });
    var shapes = [];
    for (var i = 0; i < labelYs.length; i++) {
      var want = toRgb(srcColors[i]); var k = -1, j;
      for (j = 0; j < right.length; j++) {
        if (!used[j] && toRgb(getComputedStyle(right[j]).fill) === want) { k = j; break; }
      }
      if (k < 0) { for (j = 0; j < right.length; j++) { if (!used[j]) { k = j; break; } } }
      used[k] = true;
      var b = right[k].getBoundingClientRect();
      var x0 = (b.right - svgBox.left - sz.l) / sz.w;
      var y0 = 1 - (((b.top + b.bottom) / 2) - svgBox.top - sz.t) / sz.h;
      shapes.push({type: 'line', xref: 'paper', yref: 'paper',
                   x0: x0, y0: y0, x1: labelX - 0.004, y1: labelYs[i], line: lineStyle});
    }
    gd.__leadersDone = true;
    Plotly.relayout(gd, {shapes: shapes});
  }
  gd.on('plotly_afterplot', drawLeaders);
  setTimeout(drawLeaders, 300);  // wait until Plotly's grow-in animation has finished)
})();
"""
leader_js = (leader_js
    .replace("__SRC_COLORS__", json.dumps([source_category_colors.get(s, "#999999") for s in source_nodes]))
    .replace("__LABEL_YS__", json.dumps(source_label_y))
    .replace("__LABEL_X__", str(source_x + SOURCE_LABEL_OFFSET))
    .replace("__LINE_COLOR__", LEADER_LINE_COLOR)
    .replace("__LINE_WIDTH__", str(LEADER_LINE_WIDTH)))

# Layout
fig_sankey.update_layout(
    width=1600,
    height=1200,
    margin=dict(l=170, r=290, t=30, b=30),  # wider right margin for the shifted labels
    font=dict(size=12),
    hoverlabel=dict(font_size=12)
)

# Save interactive HTML
fig_sankey_html = (FIG_DIR / "inquire_csi_country-source_category_sankey.html")
pio.write_html(fig_sankey, fig_sankey_html, include_plotlyjs="cdn", post_script=leader_js)

logging.info(f"Interactive HTML Sankey diagram saved: \n{fig_sankey_html}")
minorsep()

# Display
fig_sankey.show()  # preview only: leader lines are not drawn here
if OPEN_HTML_IN_BROWSER:
    webbrowser.open(Path(fig_sankey_html).resolve().as_uri())  # opens the HTML version with leader lines

minorsep()
logging.info(f"Sankey diagram saved: \n{fig_sankey_html} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
